## LG Aimers 9기 — 모델별 OOF 재조합과 순차 보정 (Colab)

05에서 저장한 OOF를 재사용해 **LightGBM 피처 설정과 CatBoost 피처 설정을 서로 다르게 조합**한다. 모델을 다시 학습하지 않으며, 고정 65:35 비교·혼합비 진단·이전 시즌만 이용한 rolling 선택·R/F별 보정을 검증한다. 동일한 2023·2024 OOF에서 직접 고른 최적값은 낙관적일 수 있으므로 rolling 결과와 명확히 분리한다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("Colab이 아니므로 Drive 마운트를 건너뜁니다.")

candidates = [
    Path("/content/drive/MyDrive/LG_aimers_9th"),
    Path("/content/drive/MyDrive/LG Aimers 9th"),
    Path.cwd(),
]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "experiments/05b_oof_recombination.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "LG_aimers_9th 폴더 또는 experiments/05b_oof_recombination.py를 찾지 못했습니다."
    )

required = [
    PROJECT_ROOT / "src/features.py",
    PROJECT_ROOT / "src/evaluation.py",
    PROJECT_ROOT / "src/metrics.py",
    PROJECT_ROOT / "experiments/05b_oof_recombination.py",
    PROJECT_ROOT / "outputs/feature_ablation_v1/latest_run.json",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"필수 파일이 없습니다: {missing}")

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0], "| CPU threads:", os.cpu_count())
print("05b는 재학습이 없으므로 GPU가 필요하지 않습니다.")

### 1. 05 OOF 파일 사전 점검

05의 최신 실행 폴더에서 `base`, `pitchmix_shape`, `rate_gaps`, `all_new` OOF를 읽는다. 네 파일의 `row_id`, 타깃, 시즌, 경기 유형, 행 순서가 완전히 일치해야 실행할 수 있다.

In [ ]:
FEATURE_OUTPUT_ROOT = PROJECT_ROOT / "outputs/feature_ablation_v1"
with (FEATURE_OUTPUT_ROOT / "latest_run.json").open(encoding="utf-8") as handle:
    feature_latest = json.load(handle)
SOURCE_RUN_DIR = Path(feature_latest["run_dir"])
required_configs = ["base", "pitchmix_shape", "rate_gaps", "all_new"]
oof_paths = {name: SOURCE_RUN_DIR / f"oof_{name}.csv.gz" for name in required_configs}
missing_oof = [str(path) for path in oof_paths.values() if not path.exists()]
if missing_oof:
    raise FileNotFoundError(
        f"05 OOF가 없습니다: {missing_oof}. 05 노트북 실행 결과 폴더를 지우지 않았는지 확인하세요."
    )

print("SOURCE_RUN_DIR:", SOURCE_RUN_DIR)
for name, path in oof_paths.items():
    print(f"{name:16s} {path.stat().st_size / 1e6:8.2f} MB")

### 2. OOF 재조합 실험 실행

고정 비교는 첫 제출과 같은 65:35 logit 평균을 사용한다. Grid는 LightGBM 비중 0.20~0.90을 0.05 간격으로 확인하지만, 같은 최근 OOF에서 고른 최적값은 진단값일 뿐 최종 채택값으로 바로 사용하지 않는다. Rolling 선택과 보정은 각 holdout보다 이전 시즌 OOF만 사용한다.

In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / "outputs/oof_recombination_v1"
cmd = [
    sys.executable, str(PROJECT_ROOT / "experiments/05b_oof_recombination.py"),
    "--feature-output-root", str(FEATURE_OUTPUT_ROOT),
    "--source-run-dir", str(SOURCE_RUN_DIR),
    "--output-dir", str(OUTPUT_ROOT),
]
subprocess.run(cmd, check=True)

### 3. 실행 결과와 입력 무결성 확인

In [ ]:
latest_path = OUTPUT_ROOT / "latest_run.json"
if not latest_path.exists():
    raise FileNotFoundError("latest_run.json이 없습니다. 2번 셀을 먼저 완료하세요.")
with latest_path.open(encoding="utf-8") as handle:
    latest = json.load(handle)
RUN_DIR = Path(latest["run_dir"])
with (RUN_DIR / "summary.json").open(encoding="utf-8") as handle:
    summary = json.load(handle)

audit = summary["input_audit"]
print("RUN_DIR:", RUN_DIR)
print("aligned rows:", f"{audit['rows']:,}")
print("base reconstruction max abs difference:", audit["base_reconstruction_max_abs_difference"])
if audit["base_reconstruction_max_abs_difference"] > 1e-12:
    raise RuntimeError("05 base 예측을 정확히 재현하지 못했습니다.")

### 4. 모델별 피처를 다르게 쓴 고정 65:35 비교

가장 먼저 혼합비 튜닝 없이 피처 조합 효과만 본다. `delta_brier`는 음수일수록 base보다 좋다. 2023·2024 모두 악화되지 않은 조합만 안전 후보로 간주한다.

In [ ]:
fixed_summary = pd.read_csv(RUN_DIR / "fixed_pair_summary.csv")
display(fixed_summary[[
    "pair", "recent_brier", "recent_bss_raw", "recent_mean_gap",
    "delta_recent_brier_vs_base", "delta_recent_bss_raw_vs_base",
    "delta_brier_2023_vs_base", "delta_brier_2024_vs_base",
    "improves_recent", "safe_2023_2024",
]])
print("자동 안전 후보:", summary["selected_safe_fixed_pair"])

### 5. 고정 조합의 R·F 성능

전체 개선이 F의 큰 변화 하나에만 의존하는지 확인한다. 현재 base와 자동 선택 조합, 그리고 가설이었던 `LightGBM pitchmix + CatBoost all_new`를 함께 비교한다.

In [ ]:
fixed_metrics = pd.read_csv(RUN_DIR / "fixed_pair_metrics.csv")
focus_pairs = list(dict.fromkeys([
    "lgb_base__cat_base",
    "lgb_pitchmix_shape__cat_all_new",
    summary["selected_safe_fixed_pair"],
]))
type_comparison = fixed_metrics[
    fixed_metrics["pair"].isin(focus_pairs)
    & fixed_metrics["partition"].eq("recent_pooled")
    & fixed_metrics["scope"].isin(["all", "R", "F"])
]
display(type_comparison[[
    "pair", "scope", "n", "brier", "bss_raw",
    "roc_auc", "target_rate", "prediction_mean", "mean_gap",
]].sort_values(["scope", "brier"]).reset_index(drop=True))

### 6. 혼합비 grid — 낙관적 진단값

아래 순위는 2023·2024 OOF를 보면서 같은 OOF에 가장 잘 맞는 조합과 혼합비를 고른 결과다. 모델 다양성과 유망 구간을 찾는 용도이며, 1등 행을 그대로 최종 설정으로 쓰면 선택 편향이 생긴다.

In [ ]:
optimistic_grid = pd.read_csv(RUN_DIR / "optimistic_grid_ranking.csv")
display(optimistic_grid[[
    "candidate_id", "pair", "method", "lgbm_weight",
    "brier", "bss_raw", "mean_gap",
]].head(20))
print("optimistic best:", summary["optimistic_grid_candidate"])
print("주의: 이 값은 동일 OOF 선택 결과이므로 제출 설정 확정값이 아닙니다.")

### 7. 이전 시즌만 사용한 rolling 선택

`previous`는 바로 전 시즌, `expanding`은 사용 가능한 모든 이전 OOF에서 조합·방식·혼합비를 고른다. `expert`는 R과 F에서 서로 다른 조합을 고른다. 2022→2023의 F 구조 변화가 매우 커서 rolling 결과가 불안정하면 자동 혼합비나 전문가 선택을 최종 모델에 넣지 않는다.

In [ ]:
rolling_selection = pd.read_csv(RUN_DIR / "rolling_selection.csv")
rolling_summary = pd.read_csv(RUN_DIR / "rolling_summary.csv")
display(rolling_summary[[
    "policy", "recent_brier", "recent_bss_raw", "recent_mean_gap",
    "delta_brier_2023_vs_fixed_base", "delta_brier_2024_vs_fixed_base",
    "safe_2023_2024_vs_fixed_base",
]])
display(rolling_selection[[
    "policy", "holdout_season", "selection_scope", "selection_seasons",
    "pair", "method", "lgbm_weight", "selection_brier",
]].sort_values(["policy", "holdout_season", "selection_scope"]).reset_index(drop=True))

### 8. 이전 시즌 OOF로만 학습한 확률 보정

확률 offset과 logit intercept를 전체 또는 R/F별로 계산한다. 각 holdout의 파라미터는 이전 시즌 OOF에서만 구한다. 2023·2024가 모두 base보다 좋아진 정책만 후보이며, `none`이 선택되면 현재 데이터에서는 보정을 넣지 않는 것이 안전하다는 뜻이다.

In [ ]:
calibration_summary = pd.read_csv(RUN_DIR / "calibration_summary.csv")
safe_calibration = calibration_summary[
    calibration_summary["safe_2023_2024_vs_fixed_base"]
].sort_values(["recent_brier", "brier_2024"])
display(safe_calibration[[
    "family", "policy", "recent_brier", "recent_bss_raw",
    "recent_mean_gap", "delta_brier_2023_vs_fixed_base",
    "delta_brier_2024_vs_fixed_base",
]].head(30))
print("selected calibration pair:", summary["selected_safe_calibration_pair"])
print("selected calibration policy:", summary["selected_safe_calibration_policy"])

parameters = pd.read_csv(RUN_DIR / "calibration_parameters.csv")
selected_parameters = parameters[
    parameters["pair"].eq(summary["selected_safe_calibration_pair"])
    & parameters["policy"].eq(summary["selected_safe_calibration_policy"])
]
display(selected_parameters.reset_index(drop=True))

### 9. 예측 다양성과 paired loss

서로 다른 피처 설정의 예측 상관도를 확인한다. Paired loss는 동일 행에서 후보와 base의 제곱오차 차이를 계산하며 음수가 좋다. `naive_standard_error`는 행 간 독립을 가정한 참고치일 뿐, 경기 내 상관 때문에 통계적 유의성으로 해석하지 않는다.

In [ ]:
correlations = pd.read_csv(RUN_DIR / "component_correlations.csv")
recent_correlations = correlations[correlations["partition"].eq("recent_pooled")]
display(recent_correlations.sort_values("pearson").reset_index(drop=True))

loss_deltas = pd.read_csv(RUN_DIR / "paired_loss_deltas.csv")
selected_pair = summary["selected_safe_fixed_pair"]
display(loss_deltas[
    loss_deltas["pair"].isin([selected_pair, "lgb_pitchmix_shape__cat_all_new"])
] .sort_values(["pair", "valid_season", "scope"]).reset_index(drop=True))

### 10. 다음 학습 실험 판정

05b는 저장된 모델 예측을 재조합하는 단계이므로 CatBoost에 `game_type`을 새로 넣는 실험은 아직 수행하지 않았다. 고정 비대칭 조합과 순차 보정 결과를 확인한 뒤, 다음 학습에서는 CatBoost `game_type` 포함 여부와 post-2022 F 경기 대응을 검증해야 한다.

In [ ]:
decision = {
    "selected_safe_fixed_pair": summary["selected_safe_fixed_pair"],
    "fixed_delta_brier_vs_base": summary["selected_safe_fixed_delta_brier_vs_base"],
    "selected_rolling_policy": summary["selected_rolling_policy"],
    "selected_safe_calibration_pair": summary["selected_safe_calibration_pair"],
    "selected_safe_calibration_policy": summary["selected_safe_calibration_policy"],
    "next_training_checks": [
        "CatBoost all_new with game_type included",
        "CatBoost core features without high-cardinality context interactions",
        "R/F conditional ensemble or F-specific expert",
        "latest-season-safe calibration only if rolling CV supports it",
    ],
    "submission_ready": False,
}
decision

### 11. 검토용 결과 ZIP 생성

In [ ]:
review_names = [
    "summary.json",
    "configuration.json",
    "candidate_pairs.csv",
    "fixed_pair_summary.csv",
    "fixed_pair_metrics.csv",
    "optimistic_grid_ranking.csv",
    "blend_grid_metrics.csv",
    "rolling_summary.csv",
    "rolling_selection.csv",
    "rolling_metrics.csv",
    "calibration_summary.csv",
    "calibration_parameters.csv",
    "calibration_metrics.csv",
    "component_correlations.csv",
    "paired_loss_deltas.csv",
]
missing_review = [name for name in review_names if not (RUN_DIR / name).exists()]
if missing_review:
    raise FileNotFoundError(f"검토용 결과가 누락되었습니다: {missing_review}")

REVIEW_ZIP = OUTPUT_ROOT / "oof_recomb_review.zip"
with zipfile.ZipFile(REVIEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in review_names:
        archive.write(RUN_DIR / name, arcname=name)

with zipfile.ZipFile(REVIEW_ZIP) as archive:
    print("review files:")
    for info in archive.infolist():
        print(f"  {info.filename}: {info.file_size / 1024:.1f} KB")
print("\n검토용 ZIP:", REVIEW_ZIP)
print("size (MB):", round(REVIEW_ZIP.stat().st_size / 1e6, 3))